In [ ]:
# ==========================================
# 1. SETUP & LOGIN
# ==========================================
!pip install -q --upgrade pip
!pip install -q --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio wandb huggingface_hub

import os
import torch
from huggingface_hub import login, HfApi
import wandb

# 1. Login to WandB (Replace with your key if needed)
wandb.login(key="wandb_key_here")

# 2. Login to Hugging Face
# Replace with your token: https://huggingface.co/settings/tokens
write_token = "HF_TOKEN_Here"
login(token=write_token)

# Define Model & Repo
repo_name = "whisper-small-n-demo-ultimate-train-XX"
username = "wandererupak"
repo_id = f"{username}/{repo_name}"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rupaktiwari (rupaktiwari-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
# ==========================================
# 2. DATA PREP (Using Train/Val/Test Splits)
# ==========================================
from datasets import load_dataset, Audio, DatasetDict
from transformers import WhisperTokenizer, WhisperProcessor, WhisperFeatureExtractor

model_id = 'openai/whisper-small'

print("⬇️ Loading Newari dataset (n-demo-final)...")

# 1. Load the dataset (It will automatically detect train, validation, test)
dataset = load_dataset("wandererupak/n-demo-ultimate")

# 2. Standardize column names (if needed)
# We iterate through all splits to rename columns safely
for split in dataset.keys():
    if 'transcription' in dataset[split].column_names:
        dataset[split] = dataset[split].rename_column("transcription", "sentence")
    if 'utterance_id' in dataset[split].column_names: # Renaming id to avoid confusion if needed
        dataset[split] = dataset[split].rename_column("utterance_id", "id")

print(f"✅ Dataset Structure: {dataset}")

# 3. Load Processors
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language='Nepali', task='transcribe')
processor = WhisperProcessor.from_pretrained(model_id, language='Nepali', task='transcribe')

# 4. Cast Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# 5. Prepare Dataset Function
def prepare_dataset(batch):
    audio = batch["audio"]
    # Compute input features from audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # Encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("⚙️ Preprocessing dataset...")
# Apply to all splits (train, validation, test)
dataset = dataset.map(prepare_dataset, num_proc=2)

⬇️ Loading Newari dataset (n-demo-final)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/615 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/432M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

⚙️ Preprocessing dataset...


Map (num_proc=2):   0%|          | 0/4575 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/576 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/576 [00:00<?, ? examples/s]

In [3]:
# ==========================================
# 3. COLLATOR & METRICS
# ==========================================
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import jiwer

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * jiwer.wer(label_str, pred_str)
    cer = 100 * jiwer.cer(label_str, pred_str)

    return {"wer": wer, "cer": cer}

In [5]:
# ==========================================
# 4. TRAINING CONFIG
# ==========================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, set_seed
set_seed(42)
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Force language and task
model.generation_config.language = "nepali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.dropout = 0.2
model.config.attention_dropout = 0.2

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# --- DYNAMIC STEP CALCULATION FOR SAVING ---
# We want to save twice per epoch. HF Trainer doesn't natively support "0.5 epoch",
# so we calculate the exact number of steps.
batch_size = 8
grad_accum = 2
steps_per_epoch = len(dataset["train"]) // (batch_size * grad_accum)
save_steps_count = int(steps_per_epoch / 1)
if save_steps_count < 1: save_steps_count = 1 # Safety for very small datasets

print(f"ℹ️ Steps per epoch: {steps_per_epoch}")
print(f"💾 Model will be saved every {save_steps_count} steps (Twice per epoch)")

# Initialize WandB
wandb.init(project="whisper-small-n-demo-new-data-final-XX", name="n-final-public-run-new-data-final-XX", resume="allow")

training_args = Seq2SeqTrainingArguments(
    output_dir=repo_name,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=10,
    lr_scheduler_type="cosine",
    fp16=True,

    # --- LOGGING & SAVING STRATEGY ---
    logging_strategy="steps",
    logging_steps=25,

    eval_strategy="steps",      # Evaluate every time we save
    eval_steps=save_steps_count,

    save_strategy="steps",      # Save based on steps (calculated above)
    save_steps=save_steps_count,
    save_total_limit=2,         # Keep only last 2 checkpoints to save space locally

    # --- HUB SETTINGS (PUBLIC) ---
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_private_repo=False,     # <--- CHANGED TO PUBLIC
    hub_strategy="checkpoint",  # Push every checkpoint to Hub

    # --- GENERATION ---
    predict_with_generate=True,
    generation_max_length=225,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"], # Using Validation split for training evaluation
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor, # Updated for newer transformers versions
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

print("🚀 Starting Training...")
trainer.train()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

ℹ️ Steps per epoch: 285
💾 Model will be saved every 285 steps (Twice per epoch)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Step,Training Loss,Validation Loss,Wer,Cer
285,1.180858,0.557665,71.102362,26.936100
570,0.839511,0.462493,61.692913,22.649665
855,0.553808,0.442615,58.740157,23.247605
1140,0.348267,0.444547,53.385827,20.135437
1425,0.177945,0.491222,52.952756,18.939558
1710,0.081533,0.535555,52.362205,18.817088
1995,0.027332,0.576183,51.850394,18.730639
2280,0.013092,0.614866,51.811024,18.723435
2565,0.004902,0.636950,51.417323,18.629782
2850,0.004366,0.640881,51.574803,18.709027


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=2860, training_loss=0.3904264170897903, metrics={'train_runtime': 14596.4485, 'train_samples_per_second': 3.134, 'train_steps_per_second': 0.196, 'total_flos': 1.320278206464e+19, 'train_loss': 0.3904264170897903, 'epoch': 10.0})

In [6]:
# ==========================================
# 5. TEST EVALUATION & FINAL PUSH
# ==========================================

# 1. Run Evaluation on the HELD-OUT TEST Set
print("\n🧪 Running evaluation on Test set...")
test_metrics = trainer.predict(dataset["test"]).metrics
print(f"🏆 Final Test Metrics: {test_metrics}")

# 2. Push Final Model to Hub
print("\n☁️ Pushing final model to Hugging Face Hub (PUBLIC)...")
kwargs = {
    "dataset_tags": "wandererupak/n-demo-final",
    "dataset": "N Demo Final",
    "language": "ne",
    "model_name": "Whisper Small N - Final",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)
processor.push_to_hub(repo_id)
print(f"✅ DONE! Your public model is live at: https://huggingface.co/{repo_id}")


🧪 Running evaluation on Test set...


🏆 Final Test Metrics: {'test_loss': 0.6169005632400513, 'test_wer': 51.44151565074135, 'test_cer': 18.76391554702495, 'test_runtime': 237.2526, 'test_samples_per_second': 2.428, 'test_steps_per_second': 0.303}

☁️ Pushing final model to Hugging Face Hub (PUBLIC)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rain-XX/training_args.bin: 100%|##########| 5.46kB / 5.46kB            

  ...rain-XX/model.safetensors:   3%|3         | 33.6MB /  967MB            

README.md: 0.00B [00:00, ?B/s]

✅ DONE! Your public model is live at: https://huggingface.co/wandererupak/whisper-small-n-demo-ultimate-train-XX
